# Week 1: Python & NumPy Mastery
## Notebook 08: Out-of-Core Processing with np.memmap & Virtual Memory

### 1. Conceptual Intuition & Theoretical Background
When machine learning datasets exceed available physical RAM (e.g., 50GB dataset on a 16GB RAM laptop), standard `np.load` or `pd.read_csv` crashes with an `OutOfMemoryError` (OOM).

**Memory-mapped files (`np.memmap`)** solve this by mapping a file on disk directly into the application's virtual address space via the operating system's `mmap()` system call. Rather than reading the entire array into RAM up front:
1. The OS sets up virtual page table entries pointing to the file blocks.
2. When your code accesses a slice (e.g., `data[1000:2000]`), a **page fault** occurs.
3. The OS page cache transparently fetches only the requested 4KB memory pages from disk into RAM.
4. Dirty pages (modified slices) are lazily flushed back to disk when requested or during system eviction.

$$P = \lceil \frac{\text{Array Size in Bytes}}{\text{OS Page Size (4096 bytes)}} \rceil$$

---

### 2. Scratch Python Implementation: Large Dataset Streaming

In [ ]:
import numpy as np
import os
import tempfile

# Setup temporary binary file on disk
temp_dir = tempfile.gettempdir()
memmap_path = os.path.join(temp_dir, 'large_matrix_sample.dat')

rows, cols = 50000, 200
dtype = np.float32
element_size = np.dtype(dtype).itemsize
total_bytes = rows * cols * element_size

print(f"Dataset Dimensions: {rows} x {cols}")
print(f"Total Theoretical Size: {total_bytes / (1024 * 1024):.2f} MB")

# Step 1: Initialize writable memory-mapped array on disk
fp_write = np.memmap(memmap_path, dtype=dtype, mode='w+', shape=(rows, cols))

# Step 2: Stream data generation in small chunks without loading full array in RAM
chunk_size = 10000
for i in range(0, rows, chunk_size):
    chunk_rows = min(chunk_size, rows - i)
    # Generate batch
    batch_data = np.random.randn(chunk_rows, cols).astype(dtype) + (i * 0.01)
    fp_write[i:i + chunk_rows, :] = batch_data

# Ensure all dirty pages are written to persistent disk storage
fp_write.flush()
del fp_write  # Close pointer

# Step 3: Read-only memory mapping for streaming batch inference
fp_read = np.memmap(memmap_path, dtype=dtype, mode='r', shape=(rows, cols))

# Compute streaming statistics across column 0
batch_means = []
for i in range(0, rows, chunk_size):
    # Only this slice is loaded into OS RAM page cache
    chunk = fp_read[i:i + chunk_size, 0]
    batch_means.append(np.mean(chunk))

overall_col0_mean = np.mean(batch_means)
print(f"Streaming Column 0 Mean computed: {overall_col0_mean:.4f}")

# Cleanup file
del fp_read
if os.path.exists(memmap_path):
    os.remove(memmap_path)
print("Memory map file cleanly cleaned up.")

### 3. Step-by-Step Code Walkthrough
1. **`np.memmap(mode='w+')`**: Creates an uninitialized file on disk formatted as a contiguous binary array.
2. **Chunked Writing**: We populate slices `fp_write[i:i+chunk]`. The OS flushes modified memory to disk in the background.
3. **`fp.flush()`**: Forces immediate physical write of OS page cache buffers to disk.
4. **`mode='r'`**: Opens in read-only mode. Multiple processes can concurrently read the same memory-mapped array with zero memory duplication.

---

### 4. Industry Applications & Trade-offs
- **LLM Training Datasets:** Large tokenized datasets (e.g. HuggingFace `datasets` Arrow / Memmap format) use `mmap` so millions of tokens are streamed without RAM saturation.
- **Trade-off:** Random non-contiguous reads cause excessive disk thrashing due to continuous page faults. Sequential access patterns maximize read-ahead OS caching.

---

### 5. Top Interview Questions & Answers
**Q1: How does `np.memmap` differ from standard Python file I/O (`open()` and `f.read()`)?**
> **Answer:** Standard file I/O copies data from disk into kernel memory, then copies it again into Python process user space (two memory copies). `np.memmap` maps the disk file directly into the process virtual address space via OS page tables, enabling zero-copy slice access.

**Q2: What happens if multiple worker processes read the same `np.memmap` array concurrently in PyTorch DataLoader?**
> **Answer:** Because the memory mapping is read-only, all worker processes share the exact same physical OS page cache in RAM. This prevents memory duplication across multiple DataLoader worker forks.